## SQLITE: RIGHT JOIN/DROP COLUMN

In [1]:
import sqlite3
import pandas as pd


connect = sqlite3.connect("mi_bd.db")
cursor = connect.cursor()

cursor.execute("""
  CREATE TABLE IF NOT EXISTS Master_Class (
        ID     INT PRIMARY KEY,
        NOMBRE TEXT NOT NULL,
        EDAD   INT  NOT NULL,
        CIUDAD CHAR(50),
        NOTAS  FLOAT
    )
""")

datos = {
    "Luis":  (24, "Madrid",   8.5),
    "Ana":   (32, "Lugo",     6.25),
    "Juan":  (35, "Bilbao",   5.55),
    "Nuria": (51, "Alicante", 9.75)
}
for i, (nombre, (edad, ciudad, nota)) in enumerate(datos.items()):
    cursor.execute(f"""
        INSERT INTO Master_Class (ID, NOMBRE, EDAD, CIUDAD, NOTAS)
        VALUES ({i}, '{nombre}', {edad}, '{ciudad}', {nota})
    """)
connect.commit()


In [4]:
df = pd.read_sql("SELECT * FROM Master_Class", connect)
df

,ID,NOMBRE,EDAD,CIUDAD,NOTAS
0,0,Luis,24,Madrid,8.50
1,1,Ana,32,Lugo,6.25
2,2,Juan,35,Bilbao,5.55
3,3,Nuria,51,Alicante,9.75


En versiones más antiguas de Sqlite, este no admite la ejecución de RIGHT JOIN ni de ALTER TABLE ... DROP COLUMN. En general, con las versiones actuales de Python esto no ocurre y no debería preocuparnos, tampoco debería preocuparnos en general porque en SQL con otros gestores sí que funcionan ambos y las sesiones son para aprender SQL no necesariamente SQL sobre SQLite. 

Aún así, aquí os ofrezco dos variantes para trata el problema por si os lo encontraráis cuando trabajéis en vuestro ordenador o en un ordenador con versiones antiguas

### RIGHT JOIN

La solución para el RIGHT JOIN es darle la vuelta y convertirlo en un LEFT. Así de sencillo, es decir la tabla que iría la derecha (detrás del JOIN) la ponéis a la izquierda (detrás del FROM) y en vez de un RIGHT JOIN hacéis un LEFT JOIN y solucionado. Sí por eso yo nunca uso el RIGHT.

### TIRAR COLUMNAS (ALTER TABLE... DROP)

La solución para la eliminación de columnas no es tan directa como la anterior y puede ser difícil de implementar si tenemos una tabla con grandes cantidades de datos, porque implica copiar información a otra sin las columnas que queremos eliminar.

En palabras de ChatGPT:



1. **Crear una Nueva Tabla**: Crea una nueva tabla que tenga la misma estructura que la tabla original, pero sin las columnas que deseas eliminar.

2. **Copiar Datos a la Nueva Tabla**: Copia los datos de la tabla original a la nueva tabla, omitiendo las columnas que deseas eliminar.

3. **Eliminar la Tabla Original**: Una vez que hayas verificado que los datos se han copiado correctamente, puedes eliminar la tabla original.

4. **Renombrar la Nueva Tabla**: Renombra la nueva tabla con el nombre de la tabla original.

Aquí tienes un ejemplo en Python que ilustra este proceso:

```python
import sqlite3

# Conexión a la base de datos SQLite
conn = sqlite3.connect('mi_base_de_datos.db')
cursor = conn.cursor()

# Paso 1: Crear una nueva tabla
cursor.execute('''
    CREATE TABLE nueva_tabla (
        columna1 TIPO,
        columna2 TIPO,
        ...
    )
''')

# Paso 2: Copiar datos a la nueva tabla
cursor.execute('''
    INSERT INTO nueva_tabla (columna1, columna2, ...)
    SELECT columna1, columna2, ...
    FROM tabla_original
''')

# Paso 3: Eliminar la tabla original
cursor.execute('DROP TABLE tabla_original')

# Paso 4: Renombrar la nueva tabla
cursor.execute('ALTER TABLE nueva_tabla RENAME TO tabla_original')

# Guardar los cambios y cerrar la conexión
conn.commit()
conn.close()
```

En este código, reemplaza `'mi_base_de_datos.db'`, `nueva_tabla`, `tabla_original`, `columna1`, `columna2`, etc., con los nombres reales de tu base de datos, tablas y columnas. Además, asegúrate de que el tipo de dato (`TIPO`) para cada columna coincida con el de la tabla original.

Este método es efectivo pero requiere precaución, especialmente si trabajas con grandes cantidades de datos o si la integridad de los datos es crítica. Siempre es recomendable hacer una copia de seguridad de la base de datos antes de realizar operaciones de este tipo.